# 🌿🌱 CSIRO Biomass Regression using Dense Features of DINOv2

- Use the dense patch-based features extracted by the DINOv2 model
- For each patch feature vector, use a common MLP (with weight sharing) to make predictions for each patch
- Average the MLP predictions for each patch to obtain final predictions
- Uses sharpness-aware minimization (https://github.com/davda54/sam) for training (training code not shown here)
- Computes loss only on image-level labels (TODO: incorporate regularizations to make the problem more well-conditioned, or use other methods to compute loss on patch-level labels)

# Imports

In [ ]:
!pip install /kaggle/input/pip-show-protobuf/protobuf-3.20.3-py2.py3-none-any.whl

In [ ]:
import os
import re
import glob
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset, Dataset
from PIL import Image

img_dir_path = "/kaggle/input/csiro-biomass"

In [ ]:
print("Read csv data")
d_train = pd.read_csv("/kaggle/input/csiro-biomass/train.csv", parse_dates=["Sampling_Date"])
d_test = pd.read_csv("/kaggle/input/csiro-biomass/test.csv")
print("Done")

In [ ]:
sns.boxplot(d_train, x='target', y='target_name')
plt.show()

In [ ]:
sns.boxplot(d_train, x='Height_Ave_cm', y='target_name')
plt.show()

In [ ]:
sns.boxplot(d_train, x='Pre_GSHH_NDVI', y='target_name')
plt.show()

In [ ]:
d_img_metrics = d_train.groupby(by=['image_path', 'Species'], as_index=False)[['Pre_GSHH_NDVI', 'Height_Ave_cm']].max()

In [ ]:
sns.boxplot(d_img_metrics, x='Pre_GSHH_NDVI', y='Species')

In [ ]:
d_train['filename'] = d_train.image_path.apply(lambda x: re.sub("(.jpg)|(train/)", "", x))

# Load DINOv2 backbone

In [ ]:
from transformers import AutoImageProcessor, AutoModel

In [ ]:
print("Load dinov2 model")
# processor = AutoImageProcessor.from_pretrained('facebook/dinov2-giant')
processor = AutoImageProcessor.from_pretrained('/kaggle/input/dinov2/pytorch/giant/1')
model = AutoModel.from_pretrained('/kaggle/input/dinov2/pytorch/giant/1')
print("Done")

In [ ]:
model = model.cuda()

In [ ]:
model = model.eval()

In [ ]:
# Get embedding dimension
with torch.no_grad():
    dummy_input = torch.randn(1, 3, 224, 224).cuda()
    dummy_output = model(dummy_input)
    embedding_dim = dummy_output.last_hidden_state.shape[-1]
    print(f"Embedding dimension: {embedding_dim}")

Get embedding from DINOv2

In [ ]:
start = time.time()
data_img_embed = {}
for path in tqdm(d_train.image_path.unique()):
    img_path = os.path.join(img_dir_path, path)
    img = Image.open(img_path)
    
    inputs = processor(images=img, return_tensors="pt")
    inputs = inputs.to('cuda')
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    img_embedding = outputs.last_hidden_state[:, 1:, :].squeeze(0)
    img_embedding = img_embedding.mean(0)
    data_img_embed[path] = img_embedding

duration = time.time() - start
print(f"Get image embedding duration: {duration:.2f}s")

In [ ]:
img_embedding.shape

In [ ]:
d_train_pivot = d_train.pivot(index='image_path', columns='target_name', values='target').reset_index()

In [ ]:
d_train_pivot.columns.name = None

In [ ]:
train, valid = train_test_split(d_train_pivot, test_size=0.2, random_state=123)

In [ ]:
train.shape

In [ ]:
valid.shape

In [ ]:
train.head()

In [ ]:
id2target = dict((idx, col) for idx, col in enumerate(train.columns[1:]))
target2id = dict((col, idx) for idx, col in enumerate(train.columns[1:]))

In [ ]:
class CSIRO(Dataset):
    def __init__(self, train, valid, target_map, target_name, embedding):
        self.split = {
            'train': (train, len(train)),
            'valid': (valid, len(valid))
        }
        self.target_map = target_map
        self.target_name = target_name
        self.img_embedding = embedding

        self.set_split(split="train")

    def set_split(self, split='train'):
        self.data, self.length = self.split[split]

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx, 0]
        x = self.img_embedding[img_path]
        
        y = np.array(self.data.iloc[idx, 1:].tolist())

        return x, y

    def __len__(self):
        return self.length

In [ ]:
dataset = CSIRO(train, valid, target2id, 'Dry_Clover_g', data_img_embed)
data_gen = DataLoader(dataset, batch_size = 2)
x, y = next(iter(data_gen))
x = x.cuda()
y = y.cuda()

### Define regressor

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(1536, 1024),
            nn.LeakyReLU(),
            nn.Linear(1024, 512),
            nn.LeakyReLU(),
            nn.Linear(512, 256),
            nn.LeakyReLU(),
            nn.Linear(256, 5),
            nn.ReLU()
        )

    def forward(self, input_x):
        out = self.model(input_x)

        return out

In [ ]:
print("Create MLP model...")
model_regressor = MLP()
model_regressor = model_regressor.cuda()
print("Done")

In [ ]:
class WeightedMSELoss(nn.Module):
    def __init__(self):
        super(WeightedMSELoss, self).__init__()
        # Competition weights
        self.weights = torch.tensor([0.1, 0.1, 0.1, 0.5, 0.2])  # [Dry_Clover, Dry_Dead, Dry_Green, Dry_Total, GDM]
    
    def forward(self, pred, target):
        # pred, target shape: [batch_size, 5]
        mse = (pred - target) ** 2
        weighted_mse = mse * self.weights.to(pred.device)
        
        return weighted_mse.mean()

In [ ]:
print("Define optimizer and loss function...")
optimizer = optim.AdamW(model_regressor.parameters(), lr=0.001)
criterion = WeightedMSELoss()
print("Done")

In [ ]:
for epoch in range(1, 101):
    start = time.time()

    train_loss = 0
    valid_loss = 0
    
    dataset.set_split("train")
    data_gen = DataLoader(dataset, batch_size=1024, shuffle=True)
    model_regressor.train()
    for batch_index, (x, y) in enumerate(data_gen, 1):
        x = x.cuda()
        y = y.cuda()
        
        model_regressor.zero_grad()

        out = model_regressor(x)
        loss = criterion(out, y)
        
        train_loss += (loss.item() - train_loss) / batch_index

        loss.backward()
        optimizer.step()

    dataset.set_split("valid")
    data_gen = DataLoader(dataset, batch_size=1024, shuffle=True)
    model_regressor.eval()
    for batch_index, (x, y) in enumerate(data_gen, 1):
        x = x.cuda()
        y = y.cuda()
        
        with torch.no_grad():
            out = model_regressor(x)

        loss = criterion(out, y)

        valid_loss += (loss.item() - valid_loss) / batch_index

    duration = int(time.time() - start)
    print(f"Epoch: {epoch} | Time: {duration}s")
    print(f"\t Train loss: {train_loss}")
    print(f"\t Valid loss: {valid_loss}")

## Submission

In [ ]:
d_test.head()

In [ ]:
d_test.shape

In [ ]:
print("Get test set image embedding")

start = time.time()
data_img_embed = {}
for path in tqdm(d_test.image_path.unique()):
    img_path = os.path.join(img_dir_path, path)
    img = Image.open(img_path)
    
    inputs = processor(images=img, return_tensors="pt")
    inputs = inputs.to('cuda')
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    img_embedding = outputs.last_hidden_state[:, 1:, :].squeeze(0)
    img_embedding = img_embedding.mean(0)
    data_img_embed[path] = img_embedding

duration = time.time() - start
print(f"Get image embedding duration: {duration}s")

In [ ]:
start = time.time()
result_inference = []
model_regressor.eval()
for path, embedding in data_img_embed.items():

    with torch.no_grad():
        y_pred = model_regressor(embedding)

    clover, dead, green, total, gdm = y_pred
    
    result_inference.append({
        'image_path': path, 
        'Dry_Clover_g': clover.item(), 
        'Dry_Dead_g': dead.item(), 
        'Dry_Green_g': green.item(), 
        'Dry_Total_g': total.item(), 
        'GDM_g': gdm.item()
    })
duration = time.time() - start
print(f"Regressor duration on test: {duration:.2f}s")

In [ ]:
d_test_pred_raw = pd.DataFrame(result_inference)
target_names = list(target2id.keys())
d_test_pred = d_test_pred_raw.melt(id_vars='image_path', value_vars=target_names)
d_test_pred = d_test_pred.rename({'variable': 'target_name', 'value': 'target'}, axis=1)


In [ ]:
d_submission = pd.merge(left=d_test, right=d_test_pred, how='left', on=['image_path', 'target_name'])
d_submission[['sample_id', 'target']].to_csv('submission.csv', index=False)
d_submission